1. Чистый baseline: BM25 + word/char TF-IDF

Контрольный baseline. Он нужен, чтобы честно измерить прирост последующих усложнений.

In [1]:
!pip install pyarrow beautifulsoup4 lxml pymorphy3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 90.7 MB/s eta 0:00:00


In [2]:
import sys
import subprocess
import importlib.util

In [3]:
from pathlib import Path
from functools import lru_cache
from collections import defaultdict, Counter
import html
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

pd.set_option('display.max_colwidth', 140)
SEED = 42
np.random.seed(SEED)

In [4]:
ARTICLE_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/articles.f')
CALIBRATION_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/calibration.f')
TEST_PATH = Path('/kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/test.f')

articles = pd.read_feather(ARTICLE_PATH)
calibration = pd.read_feather(CALIBRATION_PATH)
test = pd.read_feather(TEST_PATH)

print('articles:', articles.shape, ARTICLE_PATH)
print('calibration:', calibration.shape, CALIBRATION_PATH)
print('test:', test.shape, TEST_PATH)

articles: (793, 3) /kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/articles.f
calibration: (500, 3) /kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/calibration.f
test: (500, 2) /kaggle/input/datasets/naruto2009/avito-bootcamp-dataset/candidate_public/candidate_data/test.f


In [5]:
import pymorphy3

POLITE_PATTERNS = [
    r'\bздравствуйте\b', r'\bдобрый\s+(день|вечер|утро)\b',
    r'\bподскажите\b', r'\bпожалуйста\b', r'\bскажите\b',
    r'\bбудьте\s+добры\b', r'\bспасибо\b',
]
BLOCK_TAGS = ['h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'p', 'li', 'th', 'td', 'label']


def normalize_text(text: str, remove_politeness: bool = False) -> str:
    text = html.unescape(str(text or '')).lower().replace('ё', 'е')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    if remove_politeness:
        for pattern in POLITE_PATTERNS:
            text = re.sub(pattern, ' ', text)
    text = re.sub(r'[^0-9a-zа-я]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def html_to_text(raw_html: str) -> str:
    soup = BeautifulSoup(str(raw_html or ''), 'lxml')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    for image in soup.find_all('img'):
        alt = image.get('alt', '')
        if alt:
            image.replace_with(' ' + alt + ' ')
    return normalize_text(soup.get_text(' ', strip=True))


def html_to_blocks(raw_html: str) -> list[str]:
    soup = BeautifulSoup(str(raw_html or ''), 'lxml')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    for image in soup.find_all('img'):
        alt = image.get('alt', '')
        if alt:
            image.replace_with(' ' + alt + ' ')

    blocks = []
    for tag in soup.find_all(BLOCK_TAGS):
        text = normalize_text(tag.get_text(' ', strip=True))
        if text and (not blocks or text != blocks[-1]):
            blocks.append(text)
    if not blocks:
        fallback = normalize_text(soup.get_text(' ', strip=True))
        if fallback:
            blocks = [fallback]
    return blocks


def make_chunks(title: str, raw_html: str, max_chars: int = 1100, overlap_blocks: int = 1) -> list[str]:
    title_clean = normalize_text(title)
    blocks = html_to_blocks(raw_html)
    chunks, current = [], []
    current_len = 0
    for block in blocks:
        if current and current_len + len(block) + 1 > max_chars:
            chunk = normalize_text(title_clean + ' ' + ' '.join(current))
            if chunk:
                chunks.append(chunk)
            current = current[-overlap_blocks:] if overlap_blocks else []
            current_len = sum(len(x) + 1 for x in current)
        current.append(block)
        current_len += len(block) + 1
    if current:
        chunk = normalize_text(title_clean + ' ' + ' '.join(current))
        if chunk:
            chunks.append(chunk)
    return list(dict.fromkeys(chunks)) or [title_clean]


morph = pymorphy3.MorphAnalyzer()


def lemmatize_token(token: str) -> str:
    token = token.lower().replace('ё', 'е')
    if token.isdigit() or len(token) <= 1:
        return token
    if not any('а' <= char <= 'я' for char in token):
        return token
    return morph.parse(token)[0].normal_form


def lemmatize_text(text: str) -> str:
    return ' '.join(lemmatize_token(token) for token in normalize_text(text).split())


def prepare_queries(series: pd.Series) -> list[str]:
    return [normalize_text(x, remove_politeness=True) for x in series.fillna('')]

In [6]:
def parse_ground_truth(value: str) -> list[int]:
    return [int(x) for x in str(value).split() if x.strip()]


def average_precision_at_k(relevant, predicted, k=10) -> float:
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score = 0.0
    hits = 0
    used = set()
    for rank, article_id in enumerate(predicted[:k], start=1):
        if article_id in relevant and article_id not in used:
            hits += 1
            score += hits / rank
            used.add(article_id)
    return score / min(len(relevant), k)


def map_at_k(ground_truths, predictions, k=10) -> float:
    return float(np.mean([
        average_precision_at_k(gt, pred, k=k)
        for gt, pred in zip(ground_truths, predictions)
    ]))


def recall_at_k(ground_truths, predictions, k=10) -> float:
    values = []
    for gt, pred in zip(ground_truths, predictions):
        gt = set(gt)
        values.append(len(gt & set(pred[:k])) / len(gt))
    return float(np.mean(values))


def scores_to_predictions(scores: np.ndarray, article_ids: np.ndarray, k: int = 10) -> list[list[int]]:
    order = np.argsort(-scores, axis=1)[:, :k]
    return [[int(article_ids[j]) for j in row] for row in order]


def reciprocal_rank_fusion(score_matrices, weights=None, rrf_k=60) -> np.ndarray:
    if weights is None:
        weights = [1.0] * len(score_matrices)
    fused = np.zeros_like(score_matrices[0], dtype=np.float32)
    n_queries, n_docs = fused.shape
    rows = np.arange(n_queries)[:, None]
    for scores, weight in zip(score_matrices, weights):
        order = np.argsort(-scores, axis=1)
        ranks = np.empty_like(order)
        ranks[rows, order] = np.arange(1, n_docs + 1)[None, :]
        fused += weight / (rrf_k + ranks)
    return fused


def validate_and_save_submission(test_df, predictions, articles_df, filename='/kaggle/working/answer.csv'):
    valid_ids = set(articles_df['article_id'].astype(int))
    answers = []
    for pred in predictions:
        deduplicated = list(dict.fromkeys(int(x) for x in pred))[:10]
        assert 1 <= len(deduplicated) <= 10
        assert set(deduplicated).issubset(valid_ids)
        answers.append(' '.join(map(str, deduplicated)))
    submission = pd.DataFrame({'query_id': test_df['query_id'].values, 'answer': answers})
    assert len(submission) == len(test_df)
    assert submission['query_id'].is_unique
    submission.to_csv(filename, index=False)
    print(f'Сохранено: {filename}, shape={submission.shape}')
    display(submission.head())
    return submission

In [7]:
class SparseBM25:
    def __init__(self, k1=1.5, b=0.75, ngram_range=(1, 1), max_features=150_000):
        self.k1 = k1
        self.b = b
        self.vectorizer = CountVectorizer(
            ngram_range=ngram_range,
            token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b',
            max_features=max_features,
        )

    def fit(self, documents):
        from scipy.sparse import csr_matrix
        counts = self.vectorizer.fit_transform(documents).tocsr().astype(np.float32)
        n_docs = counts.shape[0]
        doc_len = np.asarray(counts.sum(axis=1)).ravel()
        avg_doc_len = max(float(doc_len.mean()), 1e-9)
        document_frequency = np.asarray((counts > 0).sum(axis=0)).ravel()
        idf = np.log1p((n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        coo = counts.tocoo()
        norm_by_row = self.k1 * (1 - self.b + self.b * doc_len / avg_doc_len)
        weighted_data = idf[coo.col] * coo.data * (self.k1 + 1) / (coo.data + norm_by_row[coo.row])
        self.document_matrix = csr_matrix(
            (weighted_data, (coo.row, coo.col)), shape=counts.shape, dtype=np.float32
        )
        return self

    def score(self, queries):
        query_matrix = self.vectorizer.transform(queries).tocsr().astype(np.float32)
        query_matrix.data[:] = 1.0
        return (query_matrix @ self.document_matrix.T).toarray().astype(np.float32)

In [8]:
article_ids = articles['article_id'].astype(int).to_numpy()
clean_bodies = articles['body'].map(html_to_text)
documents = [normalize_text((title + ' ') * 3 + body[:40_000]) for title, body in zip(articles['title'], clean_bodies)]
cal_queries = prepare_queries(calibration['query_text'])
test_queries = prepare_queries(test['query_text'])
truths = calibration['ground_truth'].map(parse_ground_truth).tolist()

In [9]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True, min_df=1, max_features=140_000,
    token_pattern=r'(?u)\b[0-9a-zа-я]{2,}\b'
)
word_docs = word_vectorizer.fit_transform(documents)
word_cal = (word_vectorizer.transform(cal_queries) @ word_docs.T).toarray().astype(np.float32)
word_test = (word_vectorizer.transform(test_queries) @ word_docs.T).toarray().astype(np.float32)

char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, min_df=2, max_features=180_000
)
char_docs = char_vectorizer.fit_transform(documents)
char_cal = (char_vectorizer.transform(cal_queries) @ char_docs.T).toarray().astype(np.float32)
char_test = (char_vectorizer.transform(test_queries) @ char_docs.T).toarray().astype(np.float32)

bm25 = SparseBM25(max_features=140_000).fit(documents)
bm25_cal = bm25.score(cal_queries)
bm25_test = bm25.score(test_queries)

In [10]:
rows = []
models = {
    'word_tfidf': word_cal,
    'char_tfidf': char_cal,
    'bm25': bm25_cal,
    'RRF': reciprocal_rank_fusion([word_cal, char_cal, bm25_cal], weights=[1.0, 1.25, 1.0]),
}
for name, scores in models.items():
    predictions = scores_to_predictions(scores, article_ids)
    rows.append({'model': name, 'MAP@10': map_at_k(truths, predictions), 'Recall@10': recall_at_k(truths, predictions)})
display(pd.DataFrame(rows).sort_values('MAP@10', ascending=False))

,model,MAP@10,Recall@10
2,bm25,0.316142,0.607167
3,RRF,0.277033,0.601667
1,char_tfidf,0.217987,0.500500
0,word_tfidf,0.208411,0.474000


In [11]:
final_test_scores = reciprocal_rank_fusion([word_test, char_test, bm25_test], weights=[1.0, 1.25, 1.0])
test_predictions = scores_to_predictions(final_test_scores, article_ids)
submission = validate_and_save_submission(test, test_predictions, articles)

Сохранено: /kaggle/working/answer.csv, shape=(500, 2)


,query_id,answer
0,1,1899 2943 4394 2223 1951 4261 1901 4308 4326 2964
1,2,3149 2802 3843 4009 3168 2521 2944 4205 1923 3006
2,3,3895 3151 3888 4403 3419 1909 2919 3148 3870 1918
3,4,3149 2943 2865 2866 3006 2696 4400 4361 3748 1775
4,5,2698 3128 4214 4294 4388 3006 1901 2220 4258 4320
